# Move Analysis Calibration

This notebook re-runs the `analyze_move` calibration scenarios and plots a 3D view of:
- `entropy`
- `ply`
- `delay_seconds`

Run in the project root with the virtualenv activated. If running from VS Code, ensure `PYTHONPATH=src` is set so `krasnal` is importable.

In [ ]:
import os
import sys
from pathlib import Path

# Ensure package imports work when the notebook is launched from repo root
repo_root = Path(os.getcwd())
src_path = repo_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import torch
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from krasnal.inference.move_analysis import analyze_move

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
def make_uniform(n: int) -> torch.Tensor:
    p = torch.ones(n, dtype=torch.float32)
    return p / p.sum()

def make_peaked(n: int, peak_idx: int = 0, peak: float = 0.9) -> torch.Tensor:
    rest = (1.0 - peak) / (n - 1)
    p = torch.full((n,), rest, dtype=torch.float32)
    p[peak_idx] = peak
    return p

def make_two_peak(n: int, a: float = 0.5, b: float = 0.3) -> torch.Tensor:
    rest = max(0.0, 1.0 - a - b) / (n - 2)
    p = torch.full((n,), rest, dtype=torch.float32)
    p[0] = a
    p[1] = b
    return p

def make_random_temp(n: int, temp: float = 1.0, seed: int | None = None) -> torch.Tensor:
    rng = torch.Generator()
    if seed is not None:
        rng.manual_seed(seed)
    logits = torch.randn(n, generator=rng)
    logits = logits / float(max(1e-6, temp))
    p = torch.softmax(logits, dim=-1)
    return p

In [ ]:
vocab_n = 16
plies = [0, 5, 10, 20, 35, 40, 60]
scenarios = [
    ('uniform', make_uniform(vocab_n)),
    ('peaked', make_peaked(vocab_n, peak_idx=0, peak=0.9)),
    ('two_peak', make_two_peak(vocab_n, a=0.5, b=0.3)),
    ('rand_cold', make_random_temp(vocab_n, temp=0.2, seed=0)),
    ('rand_hot', make_random_temp(vocab_n, temp=2.0, seed=1)),
]
rows = []
for name, probs in scenarios:
    for ply in plies:
        res = analyze_move(probs, ply)
        rows.append({
            'scenario': name,
            'ply': ply,
            'entropy': res.move_dist_entropy,
            'ply_factor': res.ply_factor,
            'delay': res.delay,
            'delay_seconds': res.delay_seconds,
        })
df = pd.DataFrame(rows)
df.head()

In [ ]:
# Plot entropy vs ply vs delay_seconds as a 3D calibration view
fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')
for name, group in df.groupby('scenario'):
    group = group.sort_values('ply')
    ax.plot(group['entropy'], group['ply'], group['delay_seconds'], marker='o', label=name)
    ax.scatter(group['entropy'], group['ply'], group['delay_seconds'], s=30)
ax.set_title('entropy vs ply vs delay_seconds')
ax.set_xlabel('entropy')
ax.set_ylabel('ply')
ax.set_zlabel('delay_seconds')
ax.legend()
plt.tight_layout()
plt.show()